# 01 — Load `Dataset_Tabular.csv` and build generator-ready splits

**Input:** `Dataset_Tabular.csv` from notebook 2 (Srishti's preprocessing, run on our own Synthea run).
**Output (to Drive):** `train.parquet`, `val.parquet`, `test.parquet`, `meta.json`, `los_train.png`

Decisions:
- PLOS label = LOS > **14.26 days** (lab's mean + 2σ computed on the full inpatient population in notebook 2). Fixed constant, not recomputed on the cohort, because the cohort (patients with ≥1 PLOS stay) has inflated LOS.
- Patient-level split 70/15/15 (seed 42): no patient appears in more than one split.
- Text columns and `birthdate` dropped. Constant / all-zero-in-train columns dropped.


In [ ]:
# Cell 1 — mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/genai_synthea/run_01"
DATA = f"{BASE}/processed/Dataset_Tabular.csv"
OUT  = f"{BASE}/genai"

PLOS_THRESHOLD = 14.26   # from notebook 2 summary: LLOS threshold (mean + 2*std, full inpatient population)
SEED = 42


In [ ]:
# Cell 2 — load and inspect
import os, json
import numpy as np
import pandas as pd

os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(DATA, low_memory=False)
print("Shape:", df.shape)
print("Non-SNOMED columns:", [c for c in df.columns if not c.startswith("snomed_")])
print("Patients:", df["subject_id"].nunique(), "| Encounters:", df["hadm_id"].nunique())


In [ ]:
# Cell 3 — select features, build label
TEXT_COLS = ["hpi", "social_history", "chief_complaint", "vitals", "labs", "conditions", "medications"]
DROP_COLS = TEXT_COLS + ["birthdate"]
ID_COLS   = ["subject_id", "hadm_id"]
CAT_COLS  = ["gender", "race", "ethnicity", "admission_type"]
NUM_COLS  = ["anchor_age"]
BIN_COLS  = [c for c in df.columns if c.startswith("snomed_")]

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
CAT_COLS = [c for c in CAT_COLS if c in df.columns]

df["target"] = pd.to_numeric(df["target"], errors="coerce")
df = df.dropna(subset=["target"]).copy()
df["is_plos"] = (df["target"] > PLOS_THRESHOLD).astype(int)

df[BIN_COLS] = df[BIN_COLS].fillna(0).astype("int8")
for c in CAT_COLS:
    df[c] = df[c].fillna("unknown").astype(str)
df["anchor_age"] = pd.to_numeric(df["anchor_age"], errors="coerce")

print(f"Rows: {len(df)} | PLOS: {df['is_plos'].sum()} ({df['is_plos'].mean():.1%})")
print("Missing age:", df["anchor_age"].isna().sum())


In [ ]:
# Cell 4 — patient-level split 70/15/15
rng = np.random.default_rng(SEED)
patients = rng.permutation(np.array(sorted(df["subject_id"].astype(str).unique()), dtype=object))
n = len(patients)
train_p = set(patients[: int(0.70 * n)])
val_p   = set(patients[int(0.70 * n): int(0.85 * n)])
test_p  = set(patients[int(0.85 * n):])

assert not (train_p & val_p) and not (train_p & test_p) and not (val_p & test_p)

split = df["subject_id"].map(lambda s: "train" if s in train_p else ("val" if s in val_p else "test"))
train, val, test = df[split == "train"].copy(), df[split == "val"].copy(), df[split == "test"].copy()

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {d['subject_id'].nunique():4d} patients | {len(d):5d} encounters | "
          f"PLOS {d['is_plos'].sum():4d} ({d['is_plos'].mean():.1%}) | "
          f"LOS p50 {d['target'].median():.0f}  p95 {d['target'].quantile(.95):.0f}  max {d['target'].max():.0f}")


In [ ]:
# Cell 5 — drop columns that carry no information in TRAIN (apply same drop to val/test)
const_cat = [c for c in CAT_COLS if train[c].nunique() <= 1]
zero_bin  = [c for c in BIN_COLS if train[c].sum() == 0]
drop = const_cat + zero_bin
for d in (train, val, test):
    d.drop(columns=drop, inplace=True)

CAT_COLS = [c for c in CAT_COLS if c not in drop]
BIN_COLS = [c for c in BIN_COLS if c not in drop]
print(f"Dropped {len(const_cat)} constant categorical {const_cat} and {len(zero_bin)} all-zero condition columns")
print(f"Features kept: {len(NUM_COLS)} numeric, {len(CAT_COLS)} categorical, {len(BIN_COLS)} condition flags")


In [ ]:
# Cell 6 — PLOS tail profile (train only): what the generators must learn
tail = train[train["is_plos"] == 1]
print(f"Train PLOS rows: {len(tail)}")
print("LOS in tail: ", tail["target"].describe().round(1).to_dict())
print("Age tail vs rest: ", round(tail["anchor_age"].mean(), 1), "vs", round(train.loc[train.is_plos == 0, "anchor_age"].mean(), 1))

prev = pd.DataFrame({
    "tail_rate": tail[BIN_COLS].mean(),
    "rest_rate": train.loc[train.is_plos == 0, BIN_COLS].mean(),
})
prev["diff"] = prev["tail_rate"] - prev["rest_rate"]
print("\nConditions most over-represented in PLOS (train):")
print(prev.sort_values("diff", ascending=False).head(10).round(3).to_string())


In [ ]:
# Cell 7 — LOS histogram with threshold (train)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(train["target"], bins=60)
ax.axvline(PLOS_THRESHOLD, color="red", ls="--", label=f"PLOS threshold {PLOS_THRESHOLD} d")
ax.set_yscale("log"); ax.set_xlabel("LOS (days)"); ax.set_ylabel("encounters (log)")
ax.set_title("Train LOS distribution"); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/los_train.png", dpi=150); plt.show()


In [ ]:
# Cell 8 — save splits + metadata
cols = ID_COLS + NUM_COLS + CAT_COLS + BIN_COLS + ["target", "is_plos"]
train[cols].to_parquet(f"{OUT}/train.parquet", index=False)
val[cols].to_parquet(f"{OUT}/val.parquet", index=False)
test[cols].to_parquet(f"{OUT}/test.parquet", index=False)

meta = {
    "source": DATA, "plos_threshold": PLOS_THRESHOLD, "seed": SEED,
    "id_cols": ID_COLS, "num_cols": NUM_COLS, "cat_cols": CAT_COLS, "bin_cols": BIN_COLS,
    "target": "target", "label": "is_plos",
    "sizes": {k: {"encounters": int(len(d)), "patients": int(d["subject_id"].nunique()),
                  "plos": int(d["is_plos"].sum())} for k, d in [("train", train), ("val", val), ("test", test)]},
}
with open(f"{OUT}/meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Saved to", OUT)
print(json.dumps(meta["sizes"], indent=2))
